# Lab 06: Memory Strategies

**Goal:** Learn different strategies for managing conversation memory when history gets too long.

**What you'll learn:**
- Buffer memory: keep everything (simple but grows)
- Window memory: keep only the last N exchanges
- Summary memory: compress old messages with an LLM
- How to choose the right strategy for your use case

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatGroq(model="llama-3.3-70b-versatile")

## Sample Conversation

Imagine a user has been chatting with the HR assistant for a while.

In [ ]:
SAMPLE_CONVERSATION = [
    SystemMessage(content="You are a UniGPS HR assistant. Be concise."),
    HumanMessage(content="How many annual leave days do I get?"),
    AIMessage(content="Full-time employees get 24 days of annual leave per year."),
    HumanMessage(content="Can I carry forward unused leave?"),
    AIMessage(content="No, unused annual leave cannot be carried forward to the next financial year."),
    HumanMessage(content="What about sick leave?"),
    AIMessage(content="You get 12 sick days per year. Unused sick leave can be carried forward up to 30 days."),
    HumanMessage(content="Do I need a doctor's note for sick leave?"),
    AIMessage(content="Yes, for absences exceeding 2 consecutive days, a medical certificate is required."),
    HumanMessage(content="What's the WFH policy?"),
    AIMessage(content="Up to 3 days/week with team lead approval. Core hours 10 AM-4 PM. Friday is mandatory in-office."),
    HumanMessage(content="What about internet reimbursement?"),
    AIMessage(content="Rs 1,500/month for WFH employees. Submit your broadband bill by the 5th of each month."),
]

total_chars = sum(len(m.content) for m in SAMPLE_CONVERSATION)
print(f"Sample conversation: {len(SAMPLE_CONVERSATION)} messages, {total_chars} chars")

## Strategy 1: Buffer Memory (keep everything)

In [ ]:
def buffer_memory(messages):
    """Keep all messages — simplest approach but grows without limit."""
    return list(messages)

buffered = buffer_memory(SAMPLE_CONVERSATION)
print(f"Messages kept: {len(buffered)}")
print(f"Characters:    {sum(len(m.content) for m in buffered)}")
print("Pros: Complete context preserved")
print("Cons: Token cost grows linearly with conversation length")

## Strategy 2: Window Memory (keep last N exchanges)

In [ ]:
def window_memory(messages, window_size=3):
    """Keep the system message and the last N human-AI exchange pairs."""
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    non_system = [m for m in messages if not isinstance(m, SystemMessage)]
    # Each exchange = 2 messages (human + ai), keep last N exchanges
    keep = non_system[-(window_size * 2):]
    return system_msgs + keep

for window in [2, 3, 5]:
    windowed = window_memory(SAMPLE_CONVERSATION, window)
    chars = sum(len(m.content) for m in windowed)
    print(f"  Window={window}: {len(windowed)} messages, {chars} chars")

In [ ]:
print("Testing window=2 with follow-up question:")
windowed = window_memory(SAMPLE_CONVERSATION, window_size=2)
follow_up = windowed + [HumanMessage(content="Remind me, how much is the internet reimbursement?")]
response = llm.invoke(follow_up)
print(f"  Q: Remind me, how much is the internet reimbursement?")
print(f"  A: {response.content[:150]}")

## Strategy 3: Summary Memory (compress with LLM)

In [ ]:
def summary_memory(messages, llm):
    """Use the LLM to summarize older messages, keeping recent ones intact."""
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    non_system = [m for m in messages if not isinstance(m, SystemMessage)]

    if len(non_system) <= 4:
        return messages  # Too short to bother summarizing

    # Split: older messages to summarize, recent ones to keep verbatim
    old_messages = non_system[:-4]
    recent_messages = non_system[-4:]

    # Ask the LLM to summarize the older conversation
    conversation_text = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
        for m in old_messages
    )
    summary_prompt = (
        f"Summarize this conversation in 2-3 sentences, "
        f"capturing key facts discussed:\n\n{conversation_text}"
    )
    summary = llm.invoke([HumanMessage(content=summary_prompt)]).content

    return system_msgs + [
        SystemMessage(content=f"Previous conversation summary: {summary}"),
    ] + recent_messages

summarized = summary_memory(SAMPLE_CONVERSATION, llm)
summary_chars = sum(len(m.content) for m in summarized)
print(f"Original:      {len(SAMPLE_CONVERSATION)} messages, {total_chars} chars")
print(f"After summary: {len(summarized)} messages, {summary_chars} chars")
print(f"Compression:   {(1 - summary_chars/total_chars)*100:.0f}% reduction")

In [ ]:
# Show the generated summary
for m in summarized:
    if isinstance(m, SystemMessage) and "summary" in m.content.lower():
        print(f"Generated summary: {m.content[:200]}")

# Test it
follow_up = summarized + [HumanMessage(content="Based on what we discussed, can I carry forward my sick leave?")]
response = llm.invoke(follow_up)
print(f"\nQ: Can I carry forward my sick leave?")
print(f"A: {response.content[:200]}")

## Compare All Strategies

In [ ]:
buf = buffer_memory(SAMPLE_CONVERSATION)
win = window_memory(SAMPLE_CONVERSATION, 3)
summ = summarized

print(f"{'Strategy':<15} {'Messages':<10} {'Characters':<12} {'Best For'}")
print("-" * 70)
print(f"{'Buffer':<15} {len(buf):<10} {sum(len(m.content) for m in buf):<12} {'Short conversations'}")
print(f"{'Window(3)':<15} {len(win):<10} {sum(len(m.content) for m in win):<12} {'Task-focused chats'}")
print(f"{'Summary':<15} {len(summ):<10} {sum(len(m.content) for m in summ):<12} {'Long conversations'}")

## TODO 1: Window Size Experiment

Try `window_memory` with `window_size=1` and ask a follow-up question about annual leave (which was discussed early on). Does the LLM still remember? Then try `window_size=5`. What's the trade-off?

In [ ]:
# win1 = window_memory(SAMPLE_CONVERSATION, window_size=___)
# follow_up = win1 + [HumanMessage(content="How many annual leave days do I get?")]
# response = llm.invoke(follow_up)
# print(f"Window=1: {response.content[:150]}")
#
# win5 = window_memory(SAMPLE_CONVERSATION, window_size=___)
# follow_up = win5 + [HumanMessage(content="How many annual leave days do I get?")]
# response = llm.invoke(follow_up)
# print(f"Window=5: {response.content[:150]}")

## TODO 2: Adaptive Memory Selector

Build a function that **automatically picks** the best memory strategy
based on conversation length:
- ≤ 6 messages → **buffer** (keep all, it's cheap)
- 7–12 messages → **window** (keep last 3 exchanges)
- \> 12 messages → **summary** (compress old, keep recent)

Test it with conversations of different lengths to verify switching.

In [ ]:
# TODO: Uncomment and build the adaptive memory selector
# def adaptive_memory(messages, llm):
#     """Pick the best memory strategy based on conversation length."""
#     non_system = [m for m in messages if not isinstance(m, SystemMessage)]
#     msg_count = len(non_system)
#
#     if msg_count <= 6:
#         strategy = "buffer"
#         result = buffer_memory(messages)
#     elif msg_count <= 12:
#         strategy = "window"
#         result = window_memory(messages, window_size=3)
#     else:
#         strategy = "summary"
#         result = summary_memory(messages, llm)
#
#     result_chars = sum(len(m.content) for m in result)
#     print(f"  Messages: {len(messages)} → Strategy: {strategy.upper()} → Output: {len(result)} msgs, {result_chars} chars")
#     return result
#
# # Test with short conversation (should use buffer)
# short_convo = SAMPLE_CONVERSATION[:5]
# print("Short conversation:")
# adaptive_memory(short_convo, llm)
#
# # Test with medium conversation (should use window)
# medium_convo = SAMPLE_CONVERSATION[:9]
# print("Medium conversation:")
# adaptive_memory(medium_convo, llm)
#
# # Test with full conversation (should use summary)
# print("Full conversation:")
# adaptive_memory(SAMPLE_CONVERSATION, llm)

## Key Takeaways

- **Buffer:** simple, complete — good for short conversations
- **Window:** fixed cost, loses old context — good for task chats
- **Summary:** balanced, adds LLM call — good for long conversations
- Choose based on: conversation length, context needs, token budget